# XLCoST boolean pipeline — Colab

Runtime → **GPU** (L4 on Pro). Three cells: setup, run, save.

The run cell is a single `scripts/run_language.sh` call that chains
gate → corpus → occurrences → extraction → probe → capped baselines with
explicit paths. Every step is idempotent or resumable — re-running the
cell after an interruption continues where it stopped.

For another language/model, edit the variables in cell 2 and re-run cells 2–3.
Never edit code here: change it in the repo, push, and re-run cell 1
(it pulls latest and prints the commit you are on).

In [ ]:
# 1 — setup: clone/pull, deps, token, restore prior work from Drive
# Reproduction/audit runs: set PIN_COMMIT to a full SHA to run vetted code
# instead of the moving branch (the branch is the default for active work
# on this private repo). Use a READ-only HF token in Colab secrets.
PIN_COMMIT = ""  # e.g. "d7ac7a9..."
import os, pathlib
if not pathlib.Path("/content/mech-interp-coding-llms").exists():
    !git clone -q -b main https://github.com/nolanlwin/mech-interp-coding-llms.git /content/mech-interp-coding-llms
%cd /content/mech-interp-coding-llms
!git fetch -q origin
_old = !git rev-parse HEAD
if PIN_COMMIT:
    !git checkout -q {PIN_COMMIT}
else:
    # A reused runtime may still be on an old branch; always land on main.
    !git checkout -q main 2>/dev/null || git checkout -q -b main origin/main
    !git pull -q
_new = !git rev-parse HEAD
if _old[0] != _new[0]:
    print("=" * 70)
    print("CODE CHANGED since this runtime last ran — review before trusting")
    print("the run with your HF token / Drive. New commits:")
    !git log --oneline {_old[0]}..{_new[0]}
    print("=" * 70)
!git log --oneline -1
!pip install -q transformers==5.8.0 tree_sitter "tree-sitter-java>=0.23.5" "tree-sitter-go>=0.25.0" \
  "tree-sitter-javascript>=0.25.0" "tree-sitter-php>=0.24.1" "tree-sitter-ruby>=0.23.1" \
  scikit-learn scipy huggingface_hub
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass
from google.colab import drive
drive.mount("/content/drive")
DEST = "/content/drive/MyDrive/mech-interp/xlcost"
!mkdir -p outputs/activations_xlcost outputs/probe_results outputs/xlcost_occ data/xlcost outputs/xlcost_occ_renamed data/xlcost_renamed
# -n: never overwrite session files; restoring lets extraction RESUME across sessions
!cp -rn {DEST}/stores/* outputs/activations_xlcost/ 2>/dev/null || true
!cp -n {DEST}/probe_results/* outputs/probe_results/ 2>/dev/null || true
!cp -n {DEST}/xlcost_occ/* outputs/xlcost_occ/ 2>/dev/null || true
!cp -n {DEST}/data_xlcost/* data/xlcost/ 2>/dev/null || true
!cp -rn {DEST}/xlcost_occ_renamed/* outputs/xlcost_occ_renamed/ 2>/dev/null || true
!cp -rn {DEST}/data_xlcost_renamed/* data/xlcost_renamed/ 2>/dev/null || true

In [ ]:
# 2 — one (language, model, split) pass, end to end
LANGUAGE = "Python"          # Python | Java | Javascript | PHP
MODEL_ID = "Qwen/Qwen2.5-Coder-1.5B"
SPLIT = "train"
!bash scripts/run_language.sh {LANGUAGE} {MODEL_ID} {SPLIT}

In [ ]:
# 3 — persist results AND the reconstructible inputs to Drive
!mkdir -p {DEST}/probe_results {DEST}/stores {DEST}/xlcost_occ {DEST}/data_xlcost
!cp -r outputs/probe_results/* {DEST}/probe_results/ 2>/dev/null || true
!cp -r outputs/xlcost_occ/* {DEST}/xlcost_occ/ 2>/dev/null || true
!cp -r data/xlcost/* {DEST}/data_xlcost/ 2>/dev/null || true
!mkdir -p {DEST}/xlcost_occ_renamed {DEST}/data_xlcost_renamed
!cp -r outputs/xlcost_occ_renamed/* {DEST}/xlcost_occ_renamed/ 2>/dev/null || true
!cp -r data/xlcost_renamed/* {DEST}/data_xlcost_renamed/ 2>/dev/null || true
!cp -r outputs/activations_xlcost/* {DEST}/stores/
!df -h /content/drive | tail -1 && ls {DEST}/probe_results/